In [88]:
print("ok")

ok


In [89]:
from langchain.agents import create_agent
from langchain.agents import AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.tools import tool, ToolRuntime
from sibyl_prompt import SIBYL_PROMPT



In [90]:
from dotenv import load_dotenv
load_dotenv()
import os
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

In [91]:

from langchain_anthropic import ChatAnthropic 
model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0,
    )

In [ ]:
client1 = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "python3",
            "args": [
                "path/to/mcp_server.py"
            ],
        }
    }
)

In [93]:
mcp_tools = await client1.get_tools()

In [94]:
from dataclasses import dataclass
@dataclass
class CustomState(AgentState):
    username: str
    academic_standing: str
    university: str

In [95]:

from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_username(username: str, runtime: ToolRuntime) -> Command:
    """ Update the username of the user in the state once they've revealed it"""
    return Command[tuple[()]](update={
        "username": username,
        "messages": [ToolMessage("Successfully updated user's username", tool_call_id=runtime.tool_call_id)]
    })
@tool
def update_user_university(university: str, runtime: ToolRuntime) -> Command:
    """ Update the university of the user in the state once they've revealed it"""
    return Command[tuple[()]](update={
        "university": university,
        "messages": [ToolMessage("Successfully updated user's university", tool_call_id=runtime.tool_call_id)]
    })
@tool
def update_user_academic_standing(academic_standing : str, runtime: ToolRuntime) -> Command:
    """ Update the academic standing of the user(like freshman, sophmore) in the state once they've revealed it"""
    return Command[tuple[()]](update={
        "academic_standing": academic_standing,
        "messages": [ToolMessage("Successfully updated user's academic standing", tool_call_id=runtime.tool_call_id)]
    })

In [96]:
agent= create_agent(
    model=model, 
    tools=[update_user_academic_standing, update_user_university,update_username, *mcp_tools],
    system_prompt=SIBYL_PROMPT,
    checkpointer=InMemorySaver(),
    state_schema= CustomState
    )

In [97]:
config={"configurable": {"thread_id":"1"}}
from langchain.messages import HumanMessage
question=HumanMessage(content="Hello. I am Oluwaferanmi Oyelude.")
response1=await agent.ainvoke({"messages": [question]}, config)

In [ ]:
from pprint import pprint
pprint(response1)

In [ ]:
question=HumanMessage(content="My name is xxxxxx, my university is xxxx University and i am a xxxx")
response2=await agent.ainvoke({"messages": [question]}, config)

In [ ]:
from pprint import pprint
pprint(response2)

In [ ]:
question=HumanMessage(content="I want you to scan my syllabi. This is the path to it: path/to/folder")
response3=await agent.ainvoke({"messages": [question]}, config)

In [ ]:
pprint(response3)

In [ ]:
pprint(response3["messages"][-2].content)

In [104]:
question=HumanMessage(content="Can you list the events you found?")
response4=await agent.ainvoke({"messages": [question]}, config)

In [ ]:
pprint(response4)

In [106]:
question=HumanMessage(content="Can you load all the high priority events to my calendar?")
response5=await agent.ainvoke({"messages": [question]}, config)

In [ ]:
pprint(response5["messages"][-2].content)

In [108]:
question=HumanMessage(content="Load the rest as well")
response6=await agent.ainvoke({"messages": [question]}, config)

In [ ]:
pprint(response6)